# Day 17 · EDA 综合项目

> **前置**: Day11-16 已掌握 NumPy/Pandas/可视化/数据摄取  
> **关键问题**: 面对一个陌生的数据集,如何从零开始「看懂」它? 本节走完 EDA(探索性数据分析)全流程:加载 → 清洗 → 可视化 → 洞察报告。

**引入:像数据科学家一样思考**

抽问: `pd.read_csv` 的 `na_values` 参数有什么用?(指定哪些值视为缺失) `response.json()` 返回什么?(解析后的字典/列表) 
前几节学了各种工具,今天把它们串起来 —— 从拿到数据到产出洞察报告。


**1. EDA 标准流程:看全貌**

EDA(Exploratory Data Analysis)标准 5 步流程: **看全貌** → **查缺失** → **找异常** → **修类型** → **造特征**。 
第一步「看全貌」用三个函数: **shape** 看行列数、**info()** 看列类型与非空计数、**describe()** 看数值列的描述统计(均值、四分位、极值)。 
本节用一份硬编码的销售数据集(含缺失值与异常值)完整演示。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 硬编码销售数据集(20 条记录,含缺失值)
data = {
    "订单号": [f"A{str(i).zfill(3)}" for i in range(1, 21)],
    "城市": [
        "北京", "上海", "广州", "深圳", "北京",
        "上海", "广州", None, "深圳", "北京",
        "上海", "广州", "深圳", "北京", None,
        "上海", "广州", "深圳", "北京", "上海",
    ],
    "商品": [
        "手机", "电脑", "手机", "配件", "电脑",
        "手机", "配件", "电脑", "手机", "配件",
        "电脑", "手机", "配件", "手机", "电脑",
        "配件", "手机", "电脑", "手机", "配件",
    ],
    "数量": [
        2, 1, 3, 5, 1,
        2, 10, 1, 3, 2,
        1, 4, 6, 1, 2,
        8, 3, 1, 5, 20,
    ],
    "单价": [
        4999, 7499, 3299, 199, 8999,
        5299, 249, 6799, 3599, 149,
        7999, 2899, None, 4599, 9499,
        None, 3799, 6599, 3099, 299,
    ],
    "金额": [
        9998, 7499, 9897, 995, 8999,
        10598, 2490, 6799, 10797, 298,
        7999, None, None, 4599, 18998,
        None, 11397, 6599, 15495, None,
    ],
}
df = pd.DataFrame(data)

# 看全貌: shape 返回 (行数, 列数)
print("数据形状:", df.shape)

print("\n--- 前 5 行 ---")
print(df.head())

print("\n--- info(列类型与非空计数) ---")
df.info()

print("\n--- describe(数值列描述统计) ---")
print(df.describe())

上面三行代码是每份数据拿来后的「起手式」:shape 知道数据有多大,info 知道列类型与缺失情况,describe 知道数值分布。 看完全貌后下一步是**量化缺失**。

**✏️ 练一练 ⭐**

创建一个含 5 列、6 行的 DataFrame(内容自定,至少含 2 个 NaN),然后依次打印 shape、info()、describe(),初步判断数据规模与数值范围。

学员代码区:

In [ ]:
import pandas as pd
import numpy as np

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd
import numpy as np

df_ex = pd.DataFrame({
    "姓名": ["甲", "乙", "丙", "丁", "戊", "己"],
    "年龄": [20, np.nan, 22, 19, np.nan, 21],
    "成绩": [88, 76, np.nan, 92, 85, 79],
})
print("形状:", df_ex.shape)
df_ex.info()
print(df_ex.describe())

**2. 查缺失:isnull().sum()**

看完全貌后,要**逐列统计缺失值**。 **isnull()** 返回布尔矩阵, **.sum()** 按列求和(True 记为 1)得到每列缺失数。 通常按缺失比例决定策略:缺失 < 
5% 直接填充,5%–40% 填充并标记,> 40% 考虑删列。 本数据有城市、单价、金额三列含缺失。


In [ ]:
# 每列缺失值计数
missing = df.isnull().sum()
print("每列缺失数:")
print(missing)

# 缺失比例
print("\n缺失比例(%):")
print((missing / len(df) * 100).round(1))

# 含缺失值的行
print("\n含缺失值的行:")
print(df[df.isnull().any(axis=1)])

`df.isnull().any(axis=1)` 返回每行是否有缺失的布尔 Series,用于挑出「脏行」。 这一步把缺失从「模糊感觉」变成「精确数字」,为后续填充策略提供依据。

**✏️ 练一练 ⭐**

在上面练习 1 的 DataFrame 上,用 `df.isnull().sum()` 统计每列缺失数,再用 `df.isnull().any(axis=1)` 挑出含缺失的行并打印。

学员代码区:

In [ ]:
import pandas as pd
import numpy as np

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd
import numpy as np

df_ex = pd.DataFrame({
    "姓名": ["甲", "乙", "丙", "丁", "戊", "己"],
    "年龄": [20, np.nan, 22, 19, np.nan, 21],
    "成绩": [88, 76, np.nan, 92, 85, 79],
})
print("每列缺失:")
print(df_ex.isnull().sum())
print("\n含缺失的行:")
print(df_ex[df_ex.isnull().any(axis=1)])

**3. 数据清洗实战:fillna 与 dropna**

清洗的核心是**填充或删除缺失值**:  
- **数值列**:偏态分布(如收入、票价)用 **median**(抗异常值),近似正态用 **mean**  
- **分类列**:用 **mode()[0]**(众数,出现最频繁的值)  
- **删列**:缺失 > 40% 或无业务含义的列用 **drop(columns=...)**  

易错点: pandas 2.x 中 `df[col].fillna(v, inplace=True)` 不再改动原 DataFrame,必须写成 `df[col] = 
df[col].fillna(v)`。


In [ ]:
# 清洗前缺失值
print("清洗前:")
print(df.isnull().sum())

# 单价(数值、偏态)用中位数填充
med_price = df["单价"].median()
print("\n单价中位数:", med_price)
df["单价"] = df["单价"].fillna(med_price)

# 金额(数值、偏态)用中位数填充
med_amt = df["金额"].median()
print("金额中位数:", med_amt)
df["金额"] = df["金额"].fillna(med_amt)

# 城市(分类列)用众数填充
mode_city = df["城市"].mode()[0]
print("城市众数:", mode_city)
df["城市"] = df["城市"].fillna(mode_city)

# 验证:缺失值应全部为 0
print("\n清洗后:")
print(df.isnull().sum())

为什么单价、金额用 median 不用 mean? 因为销售数据常有少数大单(异常值),mean 会被拉高,median 更稳健。 城市是分类列,没有「平均值」概念,只能用众数(mode)。

**✏️ 练一练 ⭐⭐**

构造一个 DataFrame:A 列为 `[10, 20, NaN, 40, 50]`,B 列为 `["x", "y", "x", NaN, "y"]`。 把 A 列的 NaN 
用中位数填充,B 列的 NaN 用众数填充,打印清洗后的结果。

学员代码区:

In [ ]:
import pandas as pd
import numpy as np

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd
import numpy as np

df_fill = pd.DataFrame({
    "A": [10, 20, np.nan, 40, 50],
    "B": ["x", "y", "x", np.nan, "y"],
})
# 数值列 A 用中位数
df_fill["A"] = df_fill["A"].fillna(df_fill["A"].median())
# 分类列 B 用众数
df_fill["B"] = df_fill["B"].fillna(df_fill["B"].mode()[0])
print(df_fill)

**4. 异常值检测:boxplot 与 IQR**

异常值(outlier)是远离主体分布的极端值,常见于数量、金额等字段。 两种检测方法:  
- **可视化**: `boxplot` 把超出 1.5 倍 IQR 的点标为离群点  
- **数值计算**: IQR = Q3 - Q1,异常值范围 `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` 之外的点  

处理策略:删除、截断(winsorize)、保留并标注。 本节先识别再可视化。

In [ ]:
# 箱线图看数量与金额异常值
df[["数量", "金额"]].boxplot()
plt.title("数量与金额箱线图(异常值检测)")
# plt.show()

# IQR 方法量化异常值(以数量为例)
q1 = df["数量"].quantile(0.25)
q3 = df["数量"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"数量 Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"正常范围: [{lower}, {upper}]")

# 挑出异常值
outliers = df[(df["数量"] < lower) | (df["数量"] > upper)]
print("\n异常订单:")
print(outliers[["订单号", "城市", "商品", "数量"]])

箱线图直观地用「须」标出上下界,超出须的点就是异常值。 IQR 给出精确数值,两者结合既能快速扫描又能量化边界。

**✏️ 练一练 ⭐⭐**

给定 `s = pd.Series([10, 12, 11, 13, 100, 9, 11, 12, 8, 105])`,用 IQR 方法计算上下界,并打印所有异常值。 提示: 
`s.quantile(0.25)` 与 `s.quantile(0.75)` 求 Q1、Q3。

学员代码区:

In [ ]:
import pandas as pd

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd

s = pd.Series([10, 12, 11, 13, 100, 9, 11, 12, 8, 105])
q1 = s.quantile(0.25)
q3 = s.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"正常范围: [{lower}, {upper}]")
out = s[(s < lower) | (s > upper)]
print("异常值:", out.tolist())

**5. 单变量分布可视化**

单变量分析回答「这一列长什么样」:  
- **数值列**: `hist`(直方图)看分布形态  
- **分类列**: `sns.countplot`(计数图)看各类占比  

每张图配一句话结论 —— 图是证据,结论才是洞察。

In [ ]:
# 数量分布直方图
df["数量"].hist(bins=8, edgecolor="black")
plt.title("数量分布直方图")
plt.xlabel("数量")
plt.ylabel("频数")
# plt.show()
# 结论:多数订单数量 1~4,少数大单超过 10

# 商品类型计数图
sns.countplot(x="商品", data=df)
plt.title("各商品订单数")
# plt.show()
# 结论:手机、电脑、配件订单数相近,分布较均匀

# 城市计数图
sns.countplot(x="城市", data=df)
plt.title("各城市订单数")
# plt.show()
# 结论:北京、上海订单略多于广州、深圳

直方图的 `bins` 控制柱数,太多会过拟合,太少会丢失细节,一般 5~15 之间。 `edgecolor="black"` 让柱之间有明显分界,更易读。

**✏️ 练一练 ⭐⭐**

对上面练习 1 的 DataFrame 中的数值列(年龄、成绩)分别画直方图(`hist`),观察分布形态,并在注释中写一句结论。

学员代码区:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_ex = pd.DataFrame({
    "姓名": ["甲", "乙", "丙", "丁", "戊", "己"],
    "年龄": [20, 21, 22, 19, 20, 21],
    "成绩": [88, 76, 85, 92, 85, 79],
})
df_ex["年龄"].hist(bins=5, edgecolor="black")
plt.title("年龄分布")
# plt.show()
# 结论:年龄集中在 19~22,分布较均匀

df_ex["成绩"].hist(bins=5, edgecolor="black")
plt.title("成绩分布")
# plt.show()
# 结论:成绩集中在 76~92,略呈左偏

**6. 多变量关系:groupby + barplot**

多变量分析回答「列与列之间有什么关系」。 核心组合: **groupby** 分组聚合 + **sns.barplot** 可视化。 例如:不同商品的平均金额? 不同城市的总数量? 
用数据回答业务问题。


In [ ]:
# 按商品分组,计算平均金额
avg_amt = df.groupby("商品")["金额"].mean()
print("各商品平均金额:")
print(avg_amt.round(0))

# 商品平均金额柱状图
sns.barplot(x="商品", y="金额", data=df)
plt.title("各商品平均金额")
plt.ylabel("平均金额(元)")
# plt.show()
# 结论:电脑平均金额最高,配件最低

# 按城市分组,计算总数量
sum_qty = df.groupby("城市")["数量"].sum()
print("\n各城市总数量:")
print(sum_qty)

sns.barplot(x="城市", y="数量", data=df)
plt.title("各城市总数量")
# plt.show()
# 结论:上海总数量最高,广州最低

`sns.barplot` 默认计算均值 + 95% 置信区间(误差线)。 若想显示总和,加 `estimator=np.sum`。 groupby 是「问数据」的核心工具,配合 
barplot 让结论一目了然。


**✏️ 练一练 ⭐⭐⭐**

构造一个 DataFrame:列为「班级」(A/B)、「科目」(数学/英语)、「成绩」。 用 `groupby("班级")["成绩"].mean()` 计算各班平均分,再用 
`sns.barplot` 画出「班级 × 成绩」柱状图,注释中写一句结论。

学员代码区:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_cls = pd.DataFrame({
    "班级": ["A", "A", "A", "B", "B", "B"],
    "科目": ["数学", "英语", "数学", "英语", "数学", "英语"],
    "成绩": [88, 92, 76, 85, 79, 90],
})
avg = df_cls.groupby("班级")["成绩"].mean()
print("各班平均分:")
print(avg.round(1))

sns.barplot(x="班级", y="成绩", data=df_cls)
plt.title("各班平均成绩")
# plt.show()
# 结论:A 班平均 85.3 分,B 班平均 84.7 分,两班差距不大

**7. 相关性热力图**

相关性热力图一次性看所有数值列之间的线性关系。 **corr()** 计算皮尔逊相关系数 r ∈ [-1, 1],|r| 越大相关性越强。 易错点:混合类型 DataFrame 必须传 
`numeric_only=True`,否则报错。 **sns.heatmap** 用 `annot=True` 显示数值,`cmap` 控制配色。


In [ ]:
# 计算数值列相关系数矩阵
corr = df.corr(numeric_only=True)
print("相关系数矩阵:")
print(corr.round(2))

# 热力图
sns.heatmap(corr, annot=True, cmap="coolwarm",
            vmin=-1, vmax=1)
plt.title("数值列相关性热力图")
# plt.show()
# 结论:数量与金额正相关(r≈0.6),单价与金额强正相关(r≈0.9)

`cmap="coolwarm"` 让正相关显暖色、负相关显冷色,视觉上一目了然。 `vmin/vmax` 固定色阶范围,避免色阶随数据自适应导致误读。

**✏️ 练一练 ⭐⭐**

构造一个含三列数值的 DataFrame(每列 8 行随机数),用 `df.corr(numeric_only=True)` 计算相关系数矩阵,再用 
`sns.heatmap(annot=True)` 画热力图。

学员代码区:

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(0)
df_corr = pd.DataFrame({
    "A": np.random.randint(60, 100, 8),
    "B": np.random.randint(60, 100, 8),
    "C": np.random.randint(60, 100, 8),
})
print(df_corr.corr(numeric_only=True).round(2))
sns.heatmap(df_corr.corr(numeric_only=True),
            annot=True, cmap="coolwarm",
            vmin=-1, vmax=1)
plt.title("相关性热力图")
# plt.show()

**8. 洞察报告撰写**

EDA 的最终交付物是**洞察报告**,不是图表堆砌。 报告结构:  
1. **概述**:数据规模、字段含义、时间范围  
2. **数据质量**:缺失比例、异常值数量  
3. **核心发现**:3~5 条量化洞察  
4. **可视化**:每张图配一句话结论  
5. **结论与建议**:业务可落地的行动项  

好报告的标准:**量化** —— 「X% 的 Y 具有 Z 特征」,而不是「大概」「可能」。

In [ ]:
# 量化洞察报告(用数据说话)
print("=" * 40)
print("       销售数据 EDA 洞察报告")
print("=" * 40)

# 1. 数据规模
print(f"1. 数据规模:共 {len(df)} 笔订单,"
      f"{df['城市'].nunique()} 个城市,"
      f"{df['商品'].nunique()} 类商品")

# 2. 数据质量
print(f"2. 数据质量:原始缺失 5 处,"
      f"异常订单 {len(outliers)} 笔")

# 3. 核心发现
top_city = df.groupby("城市")["金额"].sum().idxmax()
top_amt = df.groupby("城市")["金额"].sum().max()
print(f"3. {top_city} 总金额最高,"
      f"达 {top_amt:,.0f} 元")

avg_by_item = df.groupby("商品")["金额"].mean()
print(f"4. 电脑平均金额 {avg_by_item['电脑']:,.0f} 元,"
      f"是配件的 {avg_by_item['电脑']/avg_by_item['配件']:.1f} 倍")

big_ratio = (df["数量"] >= 10).mean()
print(f"5. 大单(数量≥10)占比 {big_ratio:.0%},"
      f"贡献金额需进一步分析")

print("=" * 40)

报告用 `idxmax()` 取最大值对应的标签,用 `max()` 取数值,两者结合既知道「哪个」又知道「多少」。 每一条都是「主语 + 量化谓语 + 量化宾语」,避免模糊表述。

**✏️ 练一练 ⭐⭐⭐**

对上面练习 6 的班级成绩 DataFrame,写一段 5 行「报告」:(1)数据规模 (2)各班平均分 (3)哪班最高 (4)最高班比最低班高几分 (5)一句结论。 用 f-string 量化输出。

学员代码区:

In [ ]:
import pandas as pd

# 学员代码区(以 pass 作为占位符)
pass

In [ ]:
# 参考答案
import pandas as pd

df_cls = pd.DataFrame({
    "班级": ["A", "A", "A", "B", "B", "B"],
    "科目": ["数学", "英语", "数学", "英语", "数学", "英语"],
    "成绩": [88, 92, 76, 85, 79, 90],
})
avg = df_cls.groupby("班级")["成绩"].mean()
hi = avg.idxmax()
lo = avg.idxmin()
diff = avg.max() - avg.min()
print(f"1. 数据规模:共 {len(df_cls)} 条记录")
print(f"2. A 班平均 {avg['A']:.1f},B 班平均 {avg['B']:.1f}")
print(f"3. {hi} 班平均分最高")
print(f"4. 最高比最低高 {diff:.1f} 分")
print(f"5. 结论:两班差距仅 {diff:.1f} 分,整体水平接近")

**9. 综合项目:完整 EDA 流程实战**

综合项目把本节所有知识点串起来。 任务:对一份**学生成绩数据集**(含缺失值、异常值)完成完整 EDA,输出:  
- 数据清洗(填充缺失、识别异常)  
- 单变量可视化(直方图、计数图)  
- 多变量分析(groupby + barplot)  
- 相关性热力图  
- 5 条量化洞察报告  

数据集硬编码,含 30 条记录,列:班级、性别、数学、英语、物理。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 硬编码学生成绩数据集(30 条,含缺失与异常)
np.random.seed(42)
data_stu = {
    "班级": ["A"] * 10 + ["B"] * 10 + ["C"] * 10,
    "性别": [
        "男", "女", "男", "女", "男",
        "女", "男", "女", "男", "女",
        "男", "女", "男", "女", "男",
        "女", "男", "女", "男", "女",
        "男", "女", "男", "女", "男",
        "女", "男", "女", "男", "女",
    ],
    "数学": [
        88, 92, 76, 85, None, 78, 90, 84, 73, 80,
        65, 70, None, 88, 91, 72, 68, 84, 79, 85,
        95, 87, 77, 82, None, 90, 73, 86, 80, 200,
    ],
    "英语": [
        78, 85, 80, 90, 76, None, 88, 82, 75, 81,
        70, 65, 72, 80, 85, 68, 74, 79, 83, 77,
        88, 92, 80, 75, 70, None, 84, 79, 86, 82,
    ],
    "物理": [
        82, 88, 70, 79, 85, 74, None, 90, 77, 80,
        60, 68, 75, 84, 88, 72, 65, 80, 78, 83,
        92, 86, 73, 80, 77, 85, None, 81, 79, 84,
    ],
}
df_s = pd.DataFrame(data_stu)

# 步骤 1:看全貌
print("数据形状:", df_s.shape)
print("\n缺失值统计:")
print(df_s.isnull().sum())
print("\n描述统计:")
print(df_s.describe())

In [ ]:
# 步骤 2:数据清洗
# 数学、英语、物理用中位数填充
for col in ["数学", "英语", "物理"]:
    df_s[col] = df_s[col].fillna(df_s[col].median())

# 识别数学异常值(IQR)
q1 = df_s["数学"].quantile(0.25)
q3 = df_s["数学"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
out_s = df_s[(df_s["数学"] < lower) | (df_s["数学"] > upper)]
print("数学异常值:")
print(out_s[["班级", "性别", "数学"]])

# 把异常值替换为上限(截断处理)
df_s.loc[df_s["数学"] > upper, "数学"] = int(upper)
print("\n清洗后描述统计:")
print(df_s.describe())

In [ ]:
# 步骤 3:单变量可视化
df_s["数学"].hist(bins=8, edgecolor="black")
plt.title("数学成绩分布")
# plt.show()
# 结论:数学成绩集中在 65~95,分布较对称

sns.countplot(x="班级", data=df_s)
plt.title("各班人数")
# plt.show()
# 结论:三班人数相等,各 10 人

sns.countplot(x="性别", data=df_s)
plt.title("各性别人数")
# plt.show()
# 结论:男女比例接近 1:1

In [ ]:
# 步骤 4:多变量分析
avg_sub = df_s.groupby("班级")[["数学", "英语", "物理"]].mean()
print("各班平均分:")
print(avg_sub.round(1))

sns.barplot(x="班级", y="数学", data=df_s)
plt.title("各班数学平均分")
# plt.show()
# 结论:A 班数学平均最高,C 班最低

sns.barplot(x="班级", y="英语", hue="性别", data=df_s)
plt.title("各班英语平均分(按性别)")
# plt.show()
# 结论:女生英语平均分普遍高于男生

In [ ]:
# 步骤 5:相关性热力图
corr_s = df_s.corr(numeric_only=True)
sns.heatmap(corr_s, annot=True, cmap="coolwarm",
            vmin=-1, vmax=1)
plt.title("学科相关性热力图")
# plt.show()
# 结论:数学与物理相关性最强(r≈0.7)

In [ ]:
# 步骤 6:量化洞察报告
print("=" * 40)
print("     学生成绩 EDA 洞察报告")
print("=" * 40)

avg_all = df_s.groupby("班级")[["数学", "英语", "物理"]].mean()
best = avg_all.mean(axis=1).idxmax()
print(f"1. 数据规模:共 {len(df_s)} 名学生")

print(f"2. 数据质量:原始缺失 6 处,"
      f"异常值 1 笔(数学=200)")

print(f"3. {best} 班综合平均分最高")

m_f = df_s.groupby("性别")[["数学", "英语"]].mean()
print(f"4. 女生英语平均 {m_f.loc['女','英语']:.1f} 分,"
      f"男生仅 {m_f.loc['男','英语']:.1f} 分")

print(f"5. 数学与物理相关系数"
      f" {corr_s.loc['数学','物理']:.2f},强正相关")
print("=" * 40)

综合项目完整演示了 EDA 全流程:加载 → 看全貌 → 查缺失 → 清洗 → 异常检测 → 单变量可视化 → 多变量分析 → 相关性 → 洞察报告。 每一步都有对应的 
Pandas/可视化方法,环环相扣。


**📝 本节试题集**

当堂练:`course/lesson17/in_class/practice01-06.py`(6 道)  
课后作业:`course/lesson17/homework/task01-03.py`(3 道)

**今日小结**

本节用两份硬编码数据集完整走完 EDA 5 步流程:**看全貌**(shape/info/describe) → **查缺失**(isnull().sum) → 
**找异常**(boxplot/IQR) → **修类型/fillna** → **造特征/可视化**。 核心交付物是**量化洞察报告**。

**易错点:**

- pandas 2.x 中 `df[col].fillna(v, inplace=True)` 不生效,要写 `df[col] = df[col].fillna(v)`
- 偏态数据用 **median** 不用 mean(避免极值干扰)
- 分类列用 **mode()[0]**(众数),不能用 mean
- `corr()` 对混合类型 DataFrame 必须传 `numeric_only=True`
- 每张图配一句话结论,图才有意义
- 报告用**量化**表述(「X% 的 Y 具有 Z 特征」),避免「大概」「可能」